In [2]:
import torch
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from huggingface_hub import login
login()

quantization_config = BitsAndBytesConfig(
    load_in_4bit = True, bnb_4bit_compute_dtype=torch.float16
)
model_name = 'meta-llama/Llama-3.1-8B'
tokenizer = AutoTokenizer.from_pretrained(model_name,quantization_config=quantization_config, device_map = 'auto')
model = AutoModel.from_pretrained(model_name, dtype = torch.float16)

# get the embedding matrix
embeddings = (model.embed_tokens.weight if hasattr(model, 'embed_tokens') else model.wte.weight) # shape -> [vocab_size, hidden_dim]
print(f'Embedding matrix shape: {embeddings.shape}')

def get_token_embedding(word):
  # get the embedding for a word (first token if multitoken)
  token_ids = tokenizer.encode(word)
  token_id = token_ids[0]
  token_text = tokenizer.decode([token_id])
  embedding = embeddings[token_id].float()
  return embedding, token_id, token_text

words = ["Python", "JavaScript", "Java", "snake", "coffee", "espresso", "tea"]
word_embeddings = {}

print("\n" + "=" * 50)
print("TOKEN EMBEDDINGS (Llama 3.1 8B)")
print("=" * 50)
for word in words:
    emb, tid, text = get_token_embedding(word)
    word_embeddings[word] = emb
    print(f"{word:12} -> token {tid:6} '{text}' -> [{emb[0]:.3f}, {emb[1]:.3f}, ...]")

def cosine_similarity(a, b):
    return torch.dot(a, b) / (torch.norm(a) * torch.norm(b))

# compute cosine similarities
print("Cosine Similarity")
pairs = [("Python", "JavaScript"), ("coffee", "espresso"),
         ("Python", "snake"), ("Java", "coffee")]
for w1, w2 in pairs:
  sim = cosine_similarity(word_embeddings[w1],word_embeddings[w2])
  print(f"sim{w1:12}, {w2:12} = {sim:.4f}")

print("\nExpected: Python-JavaScript > Python-snake")
print("Java-coffee is interesting: programming language vs the drink!")

# ============================================================
# EMBEDDING MATRIX STATS
# ============================================================
print(f"\nVocabulary size:     {embeddings.shape[0]:,}")
print(f"Embedding dimension: {embeddings.shape[1]:,}")
print(f"Total parameters:    {embeddings.numel():,}")
print(f"Memory (FP16):       {embeddings.numel() * 2 / 1e6:.1f} MB")
print(f"Memory (FP32):       {embeddings.numel() * 4 / 1e6:.1f} MB")



config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

LlamaModel LOAD REPORT from: meta-llama/Llama-3.1-8B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding matrix shape: torch.Size([128256, 4096])

TOKEN EMBEDDINGS (Llama 3.1 8B)
Python       -> token 128000 '<|begin_of_text|>' -> [0.001, -0.001, ...]
JavaScript   -> token 128000 '<|begin_of_text|>' -> [0.001, -0.001, ...]
Java         -> token 128000 '<|begin_of_text|>' -> [0.001, -0.001, ...]
snake        -> token 128000 '<|begin_of_text|>' -> [0.001, -0.001, ...]
coffee       -> token 128000 '<|begin_of_text|>' -> [0.001, -0.001, ...]
espresso     -> token 128000 '<|begin_of_text|>' -> [0.001, -0.001, ...]
tea          -> token 128000 '<|begin_of_text|>' -> [0.001, -0.001, ...]
Cosine Similarity
simPython      , JavaScript   = 1.0000
simcoffee      , espresso     = 1.0000
simPython      , snake        = 1.0000
simJava        , coffee       = 1.0000

Expected: Python-JavaScript > Python-snake
Java-coffee is interesting: programming language vs the drink!

Vocabulary size:     128,256
Embedding dimension: 4,096
Total parameters:    525,336,576
Memory (FP16):       1050.7 MB
Mem

Contextual embeddings: same word, different vectors

See how transformer layers create context-dependent representations

In [3]:
import torch
from transformers import AutoTokenizer, AutoModel

# contextual embeddings, same words different vectors eg bank (river), bank(money)
model_name = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, output_hidden_states=True)

model.eval()

def get_contextual_embeddings(sentence, target_word):
    inputs = tokenizer(sentence, return_tensors='pt')
    with torch.no_grad():
        outputs = model(**inputs)
    hidden = outputs.last_hidden_state[0]
    tokens = tokenizer.tokenize(sentence)
    for i, tok in enumerate(tokens):
        # Clean GPT-2 space prefix for matching (e.g., 'Ġbank' -> 'bank')
        cleaned = tok.replace('\u0120', ' ').strip().lower()
        if target_word.lower() is cleaned:
            return hidden[i] # GPT-2 has no BOS token, so index matches directly
    return hidden[0]

# compare bank in different contexts
contexts  = [
    ("I deposited money at the bank", "financial institution"),
    ("I walked along the river bank", "land by water"),
    ("The pilot banked the plane shortly", "tilting motion")
]

print('contextual Embeddings')
bank_emb = []
for sentence, meaning in contexts:
    emb = get_contextual_embeddings(sentence, "bank")
    bank_emb.append(emb)
    print(f"{meaning} context:")
    print(f" {sentence}")
    print(f" First five dims: [{', '.join(f'{v:3f}' for v in emb[:5])}]")

def cosine_sim(a,b):
    return torch.dot(a,b) / torch.norm(a) * torch.norm(b)


print("similarity between contextual embeddings")

for i in range(len(contexts)):
    for j in range(i + 1, len(contexts)):
        sim = cosine_sim(bank_emb[i], bank_emb[j])
        print(f"'{contexts[i][1]}' vs '{contexts[j][1]}': {sim:.4f}")

print("Embedding evolution through layers")

sentence = "the bank refused my loan application"
inputs = tokenizer(sentence, return_tensors='pt')
with torch.no_grad():
    outputs = model(**inputs)

# find bank position dynamically 
layer_tokens = tokenizer.tokenize(sentence)
bank_pos = next(i for i, t in enumerate(layer_tokens) if 'bank' in t.lower())
print(f"'bank' found at token position {bank_pos}")

prev = None

for idx, h in enumerate(outputs.hidden_states):
    emb = h[0, bank_pos]
    if prev is not None:
        change = torch.norm(emb - prev).item()
        sim = cosine_sim(emb, prev).item()
        print(f"Layer {idx:2d}: change={change:.3f}, sim_to_prev={sim:.4f}")
    else:
        print(f"Layer {idx:2d}: (initial embedding)")
    prev = emb

print("\nEarly layers change more (building context),")
print("later layers refine high-level semantics.")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


contextual Embeddings
financial institution context:
 I deposited money at the bank
 First five dims: [-0.079628, -0.065353, -0.084245, -0.033673, -0.075764]
land by water context:
 I walked along the river bank
 First five dims: [-0.079628, -0.065353, -0.084245, -0.033673, -0.075764]
tilting motion context:
 The pilot banked the plane shortly
 First five dims: [-0.047038, -0.033285, -0.162584, -0.007495, -0.097614]
similarity between contextual embeddings
'financial institution' vs 'land by water': 7273.9702
'financial institution' vs 'tilting motion': 6044.8320
'land by water' vs 'tilting motion': 6044.8320
Embedding evolution through layers
'bank' found at token position 1
Layer  0: (initial embedding)
Layer  1: change=56.659, sim_to_prev=10.0088
Layer  2: change=17.454, sim_to_prev=3213.8171
Layer  3: change=17.087, sim_to_prev=3311.5623
Layer  4: change=13.546, sim_to_prev=4157.8906
Layer  5: change=14.169, sim_to_prev=4638.7104
Layer  6: change=17.651, sim_to_prev=5528.6201
Layer